# Layer Initialization and Transfer Learning

Weights need a starting value before training begins. We can either:

- Initialize new weights
- Reuse weights learned by another model

## Layer Initialization

A linear neuron calculates a weighted sum:

```text
output = inputs * weights + bias
```

Extremely large starting weights can create unstable outputs and gradients. PyTorch therefore initializes `nn.Linear` with small random values.

In [1]:
import math
import tempfile
from pathlib import Path

import torch
from torch import nn

torch.manual_seed(42)

layer = nn.Linear(in_features=64, out_features=32)

# nn.Linear's default range depends on the number of input features
bound = 1 / math.sqrt(layer.in_features)  # 1 / sqrt(64) = 0.125

print(f"Expected bound: -{bound:.3f} to +{bound:.3f}")
print(f"Smallest weight: {layer.weight.min().item():.3f}")
print(f"Largest weight:  {layer.weight.max().item():.3f}")

Expected bound: -0.125 to +0.125
Smallest weight: -0.124
Largest weight:  0.125


The exact minimum and maximum are random, but they stay inside the calculated bound.

### Custom Initialization

`nn.init` lets us replace the default values. Here, every weight starts between `0` and `1`, while every bias starts at `0`.

In [2]:
custom_layer = nn.Linear(4, 3)

nn.init.uniform_(custom_layer.weight, a=0.0, b=1.0)  # replace weights in place
nn.init.zeros_(custom_layer.bias)                    # replace biases with zeros

print(f"Weight minimum: {custom_layer.weight.min().item():.3f}")
print(f"Weight maximum: {custom_layer.weight.max().item():.3f}")
print(f"Biases: {custom_layer.bias}")

Weight minimum: 0.061
Weight maximum: 0.924
Biases: Parameter containing:
tensor([0., 0., 0.], requires_grad=True)


## Transfer Learning

Transfer learning reuses weights learned from one task as the starting point for a related task.

![Transfer-learning workflow](diagrams/06-transfer-learning.svg)

The two models must have matching layer shapes when loading the complete set of weights.

### 1. Create and Save the Source Model

`state_dict()` contains the model's weights and biases. Saving it is the usual PyTorch approach.

In [3]:
def create_salary_model():
    return nn.Sequential(
        nn.Linear(4, 8),   # early layer learns general feature patterns
        nn.ReLU(),
        nn.Linear(8, 1),   # final layer predicts one salary value
    )


us_model = create_salary_model()

# In a real project, us_model would be trained before saving
checkpoint_path = Path(tempfile.gettempdir()) / "salary_model_weights.pth"
torch.save(us_model.state_dict(), checkpoint_path)

print(f"Saved weights to: {checkpoint_path}")

Saved weights to: C:\Users\magan\AppData\Local\Temp\salary_model_weights.pth


### 2. Load the Weights into a New Model

First create the same architecture, then load the saved `state_dict`.

In [4]:
europe_model = create_salary_model()  # starts with different random weights

saved_weights = torch.load(
    checkpoint_path,
    map_location="cpu",  # load onto the CPU
    weights_only=True,      # this file should contain weights only
)
europe_model.load_state_dict(saved_weights)  # copy saved values into the model

weights_match = torch.equal(us_model[0].weight, europe_model[0].weight)
print(f"First-layer weights match: {weights_match}")

First-layer weights match: True


## Fine-tuning

Fine-tuning continues training from the loaded weights, usually with a smaller learning rate.

We can also freeze early layers:

![Fine-tuning with a frozen early layer](diagrams/06-fine-tuning.svg)

Frozen parameters keep their loaded values because PyTorch does not calculate gradients for them.

In [5]:
# Freeze both the weight and bias of the first linear layer
for parameter in europe_model[0].parameters():
    parameter.requires_grad = False

# named_parameters() shows which values can still be trained
for name, parameter in europe_model.named_parameters():
    print(f"{name:8} | trainable: {parameter.requires_grad}")

0.weight | trainable: False
0.bias   | trainable: False
2.weight | trainable: True
2.bias   | trainable: True


### Fine-tuning Step

Only pass trainable parameters to the optimizer. A small learning rate makes gentler changes to the transferred weights.

In [6]:
europe_features = torch.randn(5, 4)
europe_salaries = torch.tensor([[55.0], [62.0], [48.0], [71.0], [59.0]])

trainable_parameters = [
    parameter
    for parameter in europe_model.parameters()
    if parameter.requires_grad
]
optimizer = torch.optim.SGD(trainable_parameters, lr=0.0001)
criterion = nn.MSELoss()  # regression loss for salary values

frozen_weight_before = europe_model[0].weight.detach().clone()
output_weight_before = europe_model[2].weight.detach().clone()

optimizer.zero_grad()  # clear gradients left from a previous step
predictions = europe_model(europe_features)          # forward pass
loss = criterion(predictions, europe_salaries)       # measure prediction error
loss.backward()   # calculate gradients for trainable parameters only
optimizer.step()  # parameter = parameter - learning rate * gradient

frozen_layer_unchanged = torch.equal(
    frozen_weight_before,
    europe_model[0].weight,
)
output_layer_updated = not torch.equal(
    output_weight_before,
    europe_model[2].weight,
)

print(f"Loss: {loss.item():.4f}")
print(f"Frozen layer unchanged: {frozen_layer_unchanged}")
print(f"Output layer updated: {output_layer_updated}")

Loss: 3538.1902
Frozen layer unchanged: True
Output layer updated: True


## Quick Look

```python
# Save
torch.save(model.state_dict(), "weights.pth")

# Load into the same architecture
model.load_state_dict(torch.load("weights.pth", weights_only=True))

# Freeze an early layer
for parameter in model[0].parameters():
    parameter.requires_grad = False

# Fine-tune the remaining parameters
optimizer = torch.optim.SGD(
    (p for p in model.parameters() if p.requires_grad),
    lr=0.0001,
)
```